# Tourist Demand Pattern Analysis Across Indian Destinations
### DS2004: Fundamentals of Data Science | Academic Course Project

**Objective:** Discover, model, and analyze tourism demand patterns across Indian destinations by integrating official Ministry of Tourism visitor statistics with authentic Open-Meteo ERA5 meteorological data.

## Phase 1: Problem Definition & Research Scope

India is endowed with profound cultural, architectural, and natural heritage. However, tourism demand is characterized by severe spatial over-concentration (top 5 monuments capture ~30% of footfall) and sharp seasonal volatility (October–March winter peaks vs. summer troughs).

**Key Research Questions:**
1. How do domestic and international tourist demand patterns differ across Indian states and destinations?
2. How do meteorological variables (temperature and precipitation) shape seasonal footfall?
3. Can Multiple Linear Regression reliably forecast tourist demand using structural and climatic variables?
4. What natural behavioral archetypes emerge when destinations are grouped using K-Means clustering?
5. What evidence-grounded interventions can relieve destination congestion and drive balanced growth?

## Phase 2 & 3: Data Ingestion & Integration

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
sys.path.append(os.path.abspath('..'))

from src.data_loader import load_all_raw_data

raw_datasets = load_all_raw_data()
for name, df in raw_datasets.items():
    print(f"{name:20s}: {df.shape[0]} rows, {df.shape[1]} columns")

## Phase 4: Data Cleaning, Standardization & Outlier Treatment

**Critical Data Quality Interventions:**
- **Aggregate Record Elimination:** The raw ASI monuments dataset contains 34 circle 'Total' and national 'Grand Total' records. Failing to remove them leads to 100% double-counting. We filter them out to isolate 144 individual monuments.
- **State Name Standardization Layer:** Cleaned inconsistent naming conventions (`Telengana` -> `Telangana`, `A&N Island` -> `Andaman & Nicobar Islands`, `J&K` -> `Jammu & Kashmir`).
- **Outlier Detection:** 14 monuments (including Taj Mahal, Qutub Minar, and Red Fort) exceed the standard IQR upper bound ($Q_3 + 1.5 \times IQR = 787,986$). We verify these as valid mega-hubs rather than errors. We retain original counts for business KPIs and create logarithmic $\ln(1 + \text{Visits})$ and 95th-percentile capped variables for linear regression modeling.

In [ ]:
from src.cleaner import clean_all_and_save

cleaned_data = clean_all_and_save()
df_dest = cleaned_data['destinations']
df_state = cleaned_data['statewise']
df_qtr = cleaned_data['quarterly']
df_fta = cleaned_data['fta']

print(f"Cleaned individual destinations: {len(df_dest)}")
print(f"Statistical upper outliers identified: {df_dest['Is_Outlier_IQR'].sum()}")
print("\nTop 5 Outliers by Volume:")
print(df_dest[df_dest['Is_Outlier_IQR']][['Monument_Name', 'Circle', 'Total_Visits_2019']].sort_values(by='Total_Visits_2019', ascending=False).head())

## Phase 5: Feature Extraction & Engineering

We formulate domain-grounded indicators:
- `Domestic_Tourist_Share` & `Foreign_Tourist_Share` (%)
- `Temp_Ideal_Deviation`: Absolute deviation from optimal thermal comfort ($22.0^\circ\text{C}$)
- `Is_UNESCO_Site`: Binary indicator for UNESCO World Heritage recognition
- `Destination_Type`: Structural/cultural classification (Fort & Palace, Temple, Caves, Mausoleum, Ruins, Memorial)
- `Monsoon_Rainfall_Share`: Proportion of annual precipitation occurring during monsoon
- `Log_Total_Visits`: Normalizing logarithmic transform

In [ ]:
from src.feature_engineering import engineer_all_and_save

df_featured = engineer_all_and_save()
print(f"Engineered feature count: {df_featured.shape[1]}")
display_cols = ['Monument_Name', 'Domestic_Tourist_Share', 'Foreign_Tourist_Share', 'Temp_Ideal_Deviation', 'Is_UNESCO_Site', 'Destination_Type']
print(df_featured[display_cols].head())

## Phase 6 & 7: Exploratory Data Analysis & Statistical Correlations

In [ ]:
# Generate all static publication-grade visualizations
import src.eda as eda

eda.generate_static_eda_figures()
print("All static figures generated and saved to outputs/figures/")

# Correlation Heatmap between key tourism and climate indicators
corr_cols = ['Total_Tourist_Visits', 'Log_Total_Visits', 'Domestic_Tourist_Share', 'Foreign_Tourist_Share', 'Annual_Avg_Temp_C', 'Annual_Total_Rainfall_mm', 'Temp_Ideal_Deviation', 'Is_UNESCO_Site']
corr_matrix = df_featured[corr_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Matrix: Tourism Demand, Tourist Mix & Weather Metrics', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Phase 8: Supervised Learning - Multiple Linear Regression

We fit an OLS Multiple Linear Regression model with target $y = \ln(1 + \text{Total Visits})$. We evaluate on a 20% holdout test set using MAE, RMSE, and $R^2$ on both the log scale and the back-transformed actual visitor scale.

In [ ]:
from src.models import train_and_save_model

regressor, reg_metrics = train_and_save_model()
print("=== REGRESSION EVALUATION METRICS ===")
for metric, val in reg_metrics.items():
    print(f"{metric:20s}: {val}")

print("\n=== STANDARDIZED REGRESSION COEFFICIENTS ===")
print(regressor.coefficients_df)

## Phase 9: Unsupervised Learning - K-Means Clustering

We segment destinations into behavioral archetypes using K-Means clustering. Optimal $k=4$ was selected based on the Elbow Method (Inertia curve) and Silhouette Analysis (score = 0.442).

In [ ]:
from src.clustering import train_and_save_clustering

clusterer, df_clustered = train_and_save_clustering()
print("=== K-MEANS VALIDATION ACROSS K ===")
print(clusterer.k_evaluation_df)

print("\n=== CLUSTER PROFILES SUMMARY ===")
print(clusterer.cluster_summary_df)

## Phase 10: Data-Driven Recommendations & Policy Insights

All recommendations are calculated directly from empirical findings:
1. **Peak Period Control:** Post-Monsoon (Q4: 30.5%) and Winter (Q1: 28.8%) demand exceeds summer trough demand by 1.6x. Timed ticketing and carrying capacity caps needed.
2. **Decongestion Imperative:** The top 5 monuments account for 29.5% of total ASI footfall. Create bundled satellite passes (e.g. Agra Fort with Fatehpur Sikri and Sikandra).
3. **Climate Resilience:** Shading and misting infrastructure for sites $>27.5^\circ\text{C}$; drainage and safety pathways for coastal sites receiving $>2500$ mm rainfall.

In [ ]:
from src.recommender import TourismRecommendationEngine

engine = TourismRecommendationEngine()
sec_rec = engine.get_seasonal_recommendations()
print("Seasonal Findings:")
print(f"Peak Season: {sec_rec['peak_season']} ({sec_rec['peak_share_pct']}%) | Trough Season: {sec_rec['trough_season']} ({sec_rec['trough_share_pct']}%)")
print(f"Seasonality Ratio: {sec_rec['seasonality_ratio']}x")

print("\nKey Decongestion Action:")
print(engine.get_decongestion_recommendations()[0])

print("\nKey Climate Action:")
print(engine.get_weather_adaptive_recommendations()[0])